# R1-M2 — production Logistic Regression: Colab verification

CPU is enough (a GPU runtime is not needed). Run all cells top to bottom. Nothing here needs the dataset.

What it checks on the **Colab** library versions (scikit-learn, librosa and NumPy differ from the local machine):
1. unpacks `src/`, `scripts/`, `tests/` from `echoproof_sync.zip` (never touches `data/` or `reports/`);
2. runs `tests/test_r2_lr_detector.py` and `tests/test_p1_08_models.py` and keeps the pytest summary line;
3. confirms NumPy scoring equals the scikit-learn pipeline within 1e-12 on generated waveforms, with and without class balancing,
   and that save -> reload is bit-exact;
4. times handcrafted-feature extraction per 4 s chunk, to plan R2-S1;
5. writes `reports/r1/r1_m2_colab_<timestamp>.json` with versions, the pytest summary and SHA-256 of every source file used.

In [ ]:
import os, sys, json, subprocess, zipfile, time, hashlib, tempfile, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

ZIP = ROOT / "echoproof_sync.zip"          # code only: src/, scripts/, tests/ - splits, manifests and reports are never overwritten
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        members = [m for m in z.namelist() if m.split("/")[0] in ("src", "scripts", "tests") and "__pycache__" not in m]
        z.extractall(ROOT, members)
    print(f"unpacked {len(members)} code files from {ZIP.name}")
SOURCES = {"lr_detector.py": ROOT / "src/echoproof/models/lr_detector.py", "acoustic.py": ROOT / "src/echoproof/features/acoustic.py",
           "ledger.py": ROOT / "src/echoproof/training/ledger.py", "trainer.py": ROOT / "src/echoproof/training/trainer.py",
           "test_r2_lr_detector.py": ROOT / "tests/test_r2_lr_detector.py"}
missing = [k for k, p in SOURCES.items() if not p.exists()]
assert not missing, f"missing in the sync zip: {missing} - rebuild echoproof_sync.zip and upload it"
SHA = {k: hashlib.sha256(p.read_bytes()).hexdigest() for k, p in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

In [ ]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, sklearn, librosa, pyarrow
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "librosa": librosa.__version__, "pyarrow": pyarrow.__version__, "platform": platform.platform()}
try:
    import torch
    ENV["torch"] = torch.__version__            # only needed by the grid runner's default metrics
except ImportError:
    ENV["torch"] = None
print(json.dumps(ENV, indent=2))

from echoproof.features.acoustic import FEATURE_NAMES, FEATURE_SCHEMA_HASH
print("feature columns:", len(FEATURE_NAMES), "| schema hash:", FEATURE_SCHEMA_HASH[:12])

## 1. Unit tests on this runtime

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r2_lr_detector.py", "tests/test_p1_08_models.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-2500:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)

## 2. NumPy scoring vs the scikit-learn pipeline, and reload parity (generated waveforms)

In [ ]:
import numpy as np
from echoproof.models.lr_detector import LRDetector, chunk_features

rng = np.random.default_rng(1)
t = np.arange(12000) / 16000
waves, y, ids = [], [], []
for r in range(12):
    label = r % 2
    for c in range(2):
        n = 12000 - 2000 * c
        f = 150 + 20 * r if label == 0 else 2500 + 30 * r
        waves.append((0.3 * np.sin(2 * np.pi * f * t[:n]) + 0.02 * rng.standard_normal(n)).astype(np.float32))
        y.append(label)
        ids.append(f"fix:{r:02d}")
y, ids = np.array(y), np.array(ids)
X = chunk_features(waves)
print("feature matrix:", X.shape, "| NaN cells:", int(np.isnan(X).sum()))

PARITY = {}
for cw in ("none", "balanced"):
    pipeline, diag = LRDetector.fit_pipeline(X, y, ids, C=1.0, class_weight=cw)
    model = LRDetector.from_pipeline(pipeline, diag, C=1.0, class_weight=cw)
    sk = pipeline.predict_proba(X)[:, 1]
    mine = model.score_chunks(waves)
    with tempfile.TemporaryDirectory() as d:
        model.save(d + "/m")
        reloaded = LRDetector.load(d + "/m").score_chunks(waves)
    PARITY[cw] = {"numpy_vs_sklearn_max_abs": float(np.max(np.abs(sk - mine))), "reload_max_abs": float(np.max(np.abs(mine - reloaded))),
                  "converged": diag["converged"], "n_iter": diag["n_iter"],
                  "mean_score_tts": float(mine[y == 1].mean()), "mean_score_genuine": float(mine[y == 0].mean())}
print(json.dumps(PARITY, indent=2))

## 3. Feature-extraction speed (for planning R2-S1)

In [ ]:
bench = [(0.3 * np.sin(2 * np.pi * (200 + 7 * i) * np.arange(64000) / 16000) + 0.05 * rng.standard_normal(64000)).astype(np.float32) for i in range(20)]
chunk_features(bench[:2])                      # warm-up (librosa/numba compilation)
start = time.perf_counter()
chunk_features(bench)
per_chunk_ms = (time.perf_counter() - start) / len(bench) * 1000
TIMING = {"ms_per_4s_chunk": round(per_chunk_ms, 1), "projected_minutes_for_20000_chunks": round(per_chunk_ms * 20000 / 60000, 1),
          "cpu_count": os.cpu_count()}
print(json.dumps(TIMING, indent=2))

## 4. Verdict and evidence

In [ ]:
TOL = 1e-12
problems = []
if not UNIT_TESTS_PASSED:
    problems.append("unit tests failed")
for cw, p in PARITY.items():
    if p["numpy_vs_sklearn_max_abs"] > TOL:
        problems.append(f"{cw}: NumPy scoring differs from sklearn by {p['numpy_vs_sklearn_max_abs']:.2e}")
    if p["reload_max_abs"] > TOL:
        problems.append(f"{cw}: reload differs by {p['reload_max_abs']:.2e}")
    if not p["converged"]:
        problems.append(f"{cw}: LR did not converge")
    if not p["mean_score_tts"] > p["mean_score_genuine"]:
        problems.append(f"{cw}: TTS fixtures do not score higher than genuine")

verdict = {"task": "R1-M2", "verified": not problems, "problems": problems, "tolerance": TOL, "pytest_summary": PYTEST_SUMMARY}
evidence = {"schema_version": 1, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "environment": ENV,
            "feature_schema_hash": FEATURE_SCHEMA_HASH, "feature_columns": len(FEATURE_NAMES), "source_sha256": SHA,
            "verdict": verdict, "parity": PARITY, "timing": TIMING}
out_dir = ROOT / "reports" / "r1"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r1_m2_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(verdict, indent=2))
print("\nevidence written to:", path)